# Exercise 2 - NLP basics
_By Georg Ahnert. Based on content by Abigail Hayes and Indira Sen._

This exercise concerns NLP basics and explores a range of methods which pre-date LLMs. We will cover the following:
- Tokenisation
- Bag-of-words and n-gram vectorization
- Classification

There will then be the opportunity to try further variants yourself.

## Setup

First we will set up our environment and prepare a dataset.

### Creating a virtual environment

If you run this notebook for the first time on the **BWUniCluster3.0**, you'll have to do some initial setup.

First, make sure that you have started a `minimal` jupyter instance as **starting an `ai` instance will likely lead to package conflicts!** Once the instance is started, open a new **Terminal** tab. Then, create a virtual environment (similar to a conda environment) and activate the environment like this:

```bash
python -m venv llm4ess_env
source llm4ess_env/bin/activate
```

Next, we need to install the minimum requirements for using our new environment as a kernel for Jupyter Notebooks. And finally, we can create a kernel based on this environment:

```bash
pip install ipykernel ipywidgets
python -m ipykernel install --user --name llm4ess --display-name "Python (LLM4ESS)" 
```

Now you can close the terminal and reload the browser window. The new kernel named `Python (LLM4ESS)` should now show up in the drop-down menu on the top right of your jupyter notebook.

You can list existing kernels with ``jupyter kernelspec list``

And remove kernels with ``jupyter kernelspec remove old_kernel``

To remve the virtual environment, simply delete the folder.

### Installing dependencies

If these Python packages are not yet available in your environment, then you should install them.
- numpy - for working with data
- pandas - for data analysis
- matplotlib - for plotting
- nltk - for natural language processing
- sklearn - install as scikit-learn - for machine learning

In [ ]:
%pip install numpy pandas matplotlib nltk scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib

### Download data

Now we will load the dataset that we are working with today. The TweetSemEval dataset contains a set of tweets and the aim is to identify if the sentiment of each tweet is positive, negative or neutral. The correct labels are already provided.

The data comes from a shared task, where researchers competed to produce the best performing models. You can also look at the [final report](https://aclanthology.org/S17-2088/) and the [GitHub repository](https://github.com/leelaylay/TweetSemEval).


In [ ]:
datapath = 'https://raw.githubusercontent.com/leelaylay/TweetSemEval/master/dataset/train/twitter-2013train-A.txt'
data = pd.read_csv(datapath, sep = '\t', names = ['id', 'sentiment', 'text'])
data

We can already see that there are 9,684 rows where each represents a tweet. For each tweet, we have an id number, the sentiment classification and the text from the tweet.

## Data exploration and preparation

We can now explore further by looking at how many tweets belong to each category:

In [ ]:
data.groupby('sentiment').size().plot(kind = 'bar')

In [ ]:
print('Full data length:', data.shape[0])

print('Unique tweets:', data['text'].nunique())

### Tokens

The text must now be broken down into tokens. Each tokenizer has different rules for some more unusual cases, but roughly equates to separating out individual words. More modern tokenizers also split subwords (e.g., "toke"-"niz"-"er"), but we'll use a basic tokenizer for now. We will use a tweet specific tokenizer so that it can handle the specific tokens that are specific to social media.

In [ ]:
from nltk.tokenize import TweetTokenizer

tt = TweetTokenizer()
# Example text
tweet = "This is a cooool #dummysmiley: :-) :-P <3 and some arrows < > -> <-- @remy: This is waaaaayyyy too much for you!!!!!!"
print(tt.tokenize(tweet))

In [ ]:
tweet = data['text'][4]
print(tt.tokenize(tweet))

From these tokens, we can now extract certain information about each tweet e.g. the number of urls.

In [ ]:
def url_count(row):
    count = 0
    tokens = tt.tokenize(row['text'])
    for token in tokens:
        if token.startswith('http'):
            count += 1
    return count

In [ ]:
url_count(data.iloc[4])

In [ ]:
data['url_count'] = data.apply(url_count, axis = 1)
data.head(5)

In [ ]:
data.groupby('url_count').size().plot(kind = 'bar')

### Train-test split

Our end goal is to build a classifier and see how well it can classify the sentiment of different tweets. As a result, we want to use some data to train the classifier and also keep some data for testing. Sklearn helps us to do this.

The `X` value is the data that will be the input to our classifier model—here the text of the tweet. The `y` value is what the model should predict—here the sentiment of the tweet.

In [ ]:
from sklearn.model_selection import train_test_split
train, test = train_test_split(data, test_size = 0.3)

In [ ]:
X_train = train['text'].values
y_train = train['sentiment'].values
X_test = test['text'].values
y_test = test['sentiment'].values

## Bag-of-words and n-grams

We cannot just provide the text data as it is. Instead we need to convert them to a vector representation i.e. a series of numerical values. You have seen various options for doing this in the lectures.

### Bag-of-words

Bag-of-words counts how often each word appears in a text. This ignores both the order and the meaning. The assumption would be that similar documents have similar counts of each word. This can also be referred to as unigrams.

**fit**: Learn a vocabulary dictionary of all tokens in the raw documents.

**transform**: Transform documents to document-term matrix.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer() # initialize the vectorizer

In [ ]:
# create the representation
vectorizer.fit(X_train)

In [ ]:
len(vectorizer.vocabulary_)

In [ ]:
bag_of_words = vectorizer.transform(X_train)
print('Bag-of-words dimensions:')
print(bag_of_words.shape)
print('Bag-of-words matrix:')
print(bag_of_words.toarray())

#### Bigrams

Rather than just looking at single words, now we will consider pairs of words. This means that some of the word order is being incorporated.

In [ ]:
bigram_vectorizer = CountVectorizer(ngram_range=(2, 2))
bigram_vectorizer.fit(X_train)
bigram_vectorizer.get_feature_names_out()

In [ ]:
bigrams = bigram_vectorizer.transform(X_train).toarray()
bigrams.shape

#### Unigrams and bigrams

We can also use both unigrams and bigrams together, increasing the number of possible combinations still further.

In [ ]:
unibigram_vectorizer = CountVectorizer(ngram_range=(1, 2))
unibigram_vectorizer.fit(X_train)
uni_bigrams = unibigram_vectorizer.transform(X_train).toarray()
uni_bigrams.shape

## Classification

Now we will build classifiers with these different vector representations. We can then compare the classifier model predictions for the test data to the correct values.

There are lots of possible models to use for classification, we will use logistic regression.

We start with the bag-of-words approach and train the model.

In [ ]:
from sklearn.linear_model import LogisticRegression
lg = LogisticRegression(random_state=0)
clf = lg.fit(bag_of_words.toarray(), y_train)

Now to make predictions on the test data.

In [ ]:
X_test_v = vectorizer.transform(X_test)
y_pred = clf.predict(X_test_v.toarray())

### Evaluation

First look at a few results.

In [ ]:
pd.DataFrame([X_test[:5], y_pred[:5], y_test[:5]]).T

In [ ]:
anti_tweet = "Ugh, this was true yesterday and it's also true now: Tom is an idiot"
clf.predict(vectorizer.transform([anti_tweet]))

Sklearn has various functions for examining results.

In [ ]:
from sklearn.metrics import classification_report # good for computing these metrics

print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_).plot()

We will store these predictions for later.

In [ ]:
results = {}
results['bow_sentiment'] = y_pred

### Compare methods

We can now repeat the process with the unigram and bigram combined vectorizer comparing the results.

In [ ]:
lg = LogisticRegression(random_state=0)
clf = lg.fit(uni_bigrams, y_train)
X_test_v = unibigram_vectorizer.transform(X_test)
y_pred = clf.predict(X_test_v.toarray())
results['unibigram_sentiment'] = y_pred

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_).plot()

## Task 1

Whether only using unigrams or also incorporating bigrams, these models have similar performance. Try the same approach with TF-IDF and see if this is better able to classify.

## Task 2

Thus far we have vectorized the text data without pre-processing. Try creating a version of the data where all the tweets have any mentions (@) and urls removed. Use this data for a combined unigrams and bigrams representation to train a logistic regression model.

 _Hint: adapt the function that we used earlier to find urls_

## Word embeddings

Packages for word embeddings are no longer as well maintained. We have a [Google Colab Jupyter Notebook](https://colab.research.google.com/drive/1saTSEzYYTrgI_gzlcTmm0zjkD-7L3opL?usp=sharing) where you can try one out. You can create your own copy and will need to install gensim in Colab first.